In [ ]:
# Filters 
import os
import sys
import torch 
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

if os.path.abspath("..") not in sys.path:
    sys.path.append(os.path.abspath(".."))

from hippy2d.learnable import flusser_basis_orders
from hippy2d.utils import get_testing_img
from hippy2d.escnn_prototype import LearnableCesa   



In [ ]:
# Simulate RGB channels
test_img_rgb = get_testing_img(rgb=True)
test_img_rgb = transforms.ToTensor()(test_img_rgb).to(dtype=torch.float64)[None, : ].to(dtype=torch.float32)
rotated_img_rgb = torch.rot90(test_img_rgb, 1, [-2, -1]).to(dtype=torch.float32)

# Plot the images
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(test_img_rgb[0].permute(1, 2, 0).numpy(), cmap='gray')
plt.title("Original Image")
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(rotated_img_rgb[0].permute(1, 2, 0).numpy(), cmap='gray')
plt.title("90 Degree Rotated Image",)
plt.axis('off')
plt.show()

# Standardize the images 
mean = test_img_rgb.mean(dim=[0, 2, 3], keepdim=True)
std = test_img_rgb.std(dim=[0, 2, 3], keepdim=True)
test_img_rgb = (test_img_rgb - mean) / std
rotated_img_rgb = (rotated_img_rgb - mean) / std

In [ ]:
from typing import Optional

from einops import rearrange

In [ ]:
# Forward pass 
layer = LearnableCesa(in_channels=3, input_size=None, out_channels=15, max_order=4, kernel_size=15)
layer.eval()
out_trivial = rearrange(layer(test_img_rgb)[0], 'b ch o h w -> (b ch o) h w') 
out_rotated_trivial = torch.rot90(rearrange(layer(rotated_img_rgb)[0], 'b ch o h w -> (b ch o) h w'), -1, [-2, -1])


# plot all the trivials My moments are parametrized by harmonics on the phasep.argwhere(self.orders ==0)[-1] and learnable rings basis. Similar as they use in escnn. My input is always real number
num_trivial = out_trivial.shape[0]
plt.figure(figsize=(15, 5))
for i in range(num_trivial):
    plt.subplot(2, num_trivial, i+1)
    plt.imshow(out_trivial[i].detach().numpy(), cmap='gray')
    plt.title(f"{i+1} (Original)")
    plt.axis('off')
    
    plt.subplot(2, num_trivial, i+1+num_trivial)
    plt.imshow(out_rotated_trivial[i].detach().numpy(), cmap='gray')
    plt.title(f"{i+1} (Rot)")
    plt.axis('off')
    torch.testing.assert_close(out_trivial[i], out_rotated_trivial[i])


In [ ]:
from hippy2d.utils import complex_to_rgb


non_trivial = rearrange(layer(test_img_rgb)[1], 'b ch o h w -> (b ch o) h w')
non_trivial_rotated = torch.rot90(rearrange(layer(rotated_img_rgb)[1], 'b ch o h w -> (b ch o) h w'), -1, [-2, -1])
    
# plot all the non-trivials and check rotation equivariance
num_non_trivial = non_trivial.shape[0]
plt.figure(figsize=(20, 7))
for i in range(num_non_trivial):
    plt.subplot(2, num_non_trivial, i+1)
    complex_img = complex_to_rgb(non_trivial[i].detach())
    plt.imshow(complex_img)
    plt.title(f"{i+1} (Org)")
    plt.axis('off')
    
    plt.subplot(2, num_non_trivial, i+1+num_non_trivial)
    complex_img_rotated = complex_to_rgb(non_trivial_rotated[i].detach())
    plt.imshow(complex_img_rotated)
    plt.title(f"{i+1} (Rot)")
    plt.axis('off')
    torch.testing.assert_close(non_trivial[i], non_trivial_rotated[i], rtol=1e-4, atol=1e-4)


In [ ]:
layer.out_type.size - layer.input_channels